# ViMMKG — trích đặc trưng (bản Colab)

Chuyển từ Kaggle sang vì hết quota GPU tuần và kernel Kaggle thất bại chưa chẩn
đoán được (endpoint tải log bị chặn tốc độ).

Việc: tải nốt ảnh còn thiếu → trích **CLIP ViT-B/32** (512-d) và
**multilingual-e5-large** (1024-d) → sinh `missing.json`.

**Mỗi giai đoạn lưu ngay vào Drive.** Colab hay ngắt; chạy lại là tiếp tục được,
không mất phần đã xong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/vimmkg"
os.makedirs(OUT, exist_ok=True)
print("kết quả sẽ nằm ở:", OUT)
print("đã có:", sorted(os.listdir(OUT)))

In [ ]:
# Cần kaggle.json để tải dataset vimmkg-raw.
# Lấy ở kaggle.com > Settings > API > Create New Token, rồi tải lên đây.
import os, json
if not os.path.exists("/root/.kaggle/kaggle.json"):
    from google.colab import files
    print("Tải lên tệp kaggle.json:")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json","wb") as f:
        f.write(next(iter(up.values())))
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip install -q kaggle 2>&1 | tail -1
print("kaggle sẵn sàng")

In [ ]:
# Tải dataset. ~219MB, Kaggle tự giải nén .tar.gz khi tạo dataset nên ảnh
# nằm sẵn dưới dạng thư mục (đã kiểm chứng bằng `kaggle datasets files`).
import os, time
IN = "/content/vimmkg_raw"
if not os.path.exists(f"{IN}/dataset/train2id.txt"):
    os.makedirs(IN, exist_ok=True)
    t0 = time.time()
    !kaggle datasets download -d nhantrnh/vimmkg-raw -p /content --unzip -q
    !mkdir -p {IN} && cp -r /content/dataset {IN}/ 2>/dev/null
    !cp -r /content/images_partial {IN}/ 2>/dev/null
    !cp /content/*.json {IN}/ 2>/dev/null
    print(f"tải xong trong {time.time()-t0:.0f}s")
print("nội dung:", sorted(os.listdir(IN)))

In [ ]:
# Dò theo TỆP MỐC thay vì đoán đường dẫn — bố cục sau giải nén có thể lồng.
import os, json, shutil
def find_file(marker, root="/content"):
    for dp, dn, fn in os.walk(root):
        if marker in fn and "/drive/" not in dp: return dp
    return None
def find_images(root="/content"):
    best, n = None, 0
    for dp, dnn, fn in os.walk(root):
        if "/drive/" in dp: continue
        c = sum(1 for f in fn if f.endswith(".jpg") and not f.startswith("._"))
        if c > n: best, n = dp, c
    return best, n

DS  = find_file("train2id.txt")
EX  = find_file("viwiki_extracts.json")
TI  = find_file("viwiki_titles.json")
P18 = find_file("p18_names.json")
IMG, N_IMG = find_images()
for lbl, v in (("dataset",DS),("extracts",EX),("titles",TI),("p18",P18),("ảnh",f"{IMG} ({N_IMG:,})")):
    print(f"  {lbl:10}: {v}")
assert all([DS,EX,TI,P18,IMG]), "thiếu dữ liệu đầu vào"

W = "/content/work"; os.makedirs(f"{W}/images", exist_ok=True)
n = 0
for f in os.listdir(IMG):
    if f.endswith(".jpg") and not f.startswith("._") and not os.path.exists(f"{W}/images/{f}"):
        shutil.copy(f"{IMG}/{f}", f"{W}/images/{f}"); n += 1
print(f"chép {n:,} ảnh, tổng {len(os.listdir(f'{W}/images')):,}")

In [ ]:
# entity list theo đúng thứ tự chỉ số
lines = open(f"{DS}/entity2id.txt").read().split("\n")
N = int(lines[0]); ents = [None]*N
for l in lines[1:]:
    if not l.strip(): continue
    q, i = l.rsplit("\t",1) if "\t" in l else l.rsplit(" ",1)
    ents[int(i)] = q
assert all(e is not None for e in ents), "entity2id.txt có lỗ hổng"
names = json.load(open(f"{P18}/p18_names.json"))
have  = {f[:-4] for f in os.listdir(f"{W}/images") if f.endswith(".jpg")}
todo  = [(q, names[q]) for q in ents if q in names and q not in have]
print(f"{N:,} thực thể | đã có {len(have):,} ảnh | còn tải {len(todo):,}")

In [ ]:
# Tải ảnh còn thiếu. Commons chặn tốc độ (HTTP 429) — cấu hình dưới đã đo thật
# trên máy: 4 luồng + nghỉ 0,5s + lùi cấp số nhân => hỏng ~1,6%.
import urllib.request, urllib.parse, urllib.error, threading, queue, collections, random, time
UA = "ViMMKG-research/0.1 (academic; your-email@example.com)"
q_ = queue.Queue()
for x in todo: q_.put(x)
lock=threading.Lock(); st=collections.Counter(); t0=time.time(); TOT=len(todo)

def work():
    while True:
        try: qid, fn = q_.get_nowait()
        except queue.Empty: return
        url = "https://commons.wikimedia.org/wiki/Special:FilePath/"+urllib.parse.quote(fn)+"?width=224"
        ok=False
        for att in range(4):
            try:
                rq = urllib.request.Request(url, headers={"User-Agent": UA})
                with urllib.request.urlopen(rq, timeout=60) as r: data=r.read()
                if len(data) > 500:
                    open(f"{W}/images/{qid}.jpg","wb").write(data); ok=True
                break
            except urllib.error.HTTPError as e:
                if e.code == 429:
                    with lock: st["429"]+=1
                    time.sleep((2**att)*5 + random.random()*3)
                elif e.code in (403,404):
                    with lock: st[f"http{e.code}"]+=1; break
                else: time.sleep(3)
            except Exception: time.sleep(3)
        with lock:
            st["ok" if ok else "fail"]+=1
            n=st["ok"]+st["fail"]
            if n % 500 == 0:
                el=time.time()-t0
                print(f"  {n:>6}/{TOT} ok={st['ok']:,} fail={st['fail']} 429={st['429']}"
                      f"  {el/60:.1f}ph còn~{(TOT-n)*el/n/60:.0f}ph", flush=True)
        time.sleep(0.5)

if TOT:
    ths=[threading.Thread(target=work,daemon=True) for _ in range(4)]
    [t.start() for t in ths]; [t.join() for t in ths]
print(f"XONG ok={st['ok']:,} fail={st['fail']:,} 429={st['429']} 404={st.get('http404',0)}")
print(f"tổng ảnh: {len(os.listdir(f'{W}/images')):,}/{N:,}")

In [ ]:
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", DEV, torch.cuda.get_device_name(0) if DEV=="cuda" else "")
assert DEV == "cuda", "Runtime > Change runtime type > GPU"

In [ ]:
# ĐẶC TRƯNG THỊ GIÁC — CLIP ViT-B/32 (512-d).
# Bỏ qua nếu đã có trên Drive (Colab hay ngắt).
import os, torch, time
from PIL import Image
VIS = f"{OUT}/ViMMKG-visual.pth"
if os.path.exists(VIS):
    out = torch.load(VIS, map_location="cpu")
    print(f"đã có: {tuple(out.shape)}")
else:
    from transformers import CLIPModel, CLIPImageProcessor
    m = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval().to(DEV)
    p = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def _img_feat(o):
        """Lấy vector bất kể phiên bản transformers.

        4.x trả thẳng Tensor; 5.x trả BaseModelOutputWithPooling với vector ở
        pooler_output (đã qua visual_projection — kiểm chứng khớp tuyệt đối với
        visual_projection(vision_model(x).pooler_output)).
        """
        if isinstance(o, torch.Tensor): return o
        if hasattr(o, "pooler_output"): return o.pooler_output
        raise TypeError(f"không rõ kiểu trả về: {type(o)}")

    out = torch.zeros(N, m.config.projection_dim)
    idx, ims = [], []; n_ok=n_bad=0; t0=time.time(); BS=128
    @torch.no_grad()
    def flush():
        global idx, ims
        if not ims: return
        x = p(images=ims, return_tensors="pt").to(DEV)
        out[idx] = _img_feat(m.get_image_features(**x)).float().cpu()
        idx, ims = [], []
    for i,q in enumerate(ents):
        f=f"{W}/images/{q}.jpg"
        if not os.path.exists(f): continue
        try: ims.append(Image.open(f).convert("RGB")); idx.append(i); n_ok+=1
        except Exception: n_bad+=1; continue
        if len(ims)>=BS:
            flush()
            if n_ok % 2048 < BS: print(f"  {n_ok:>6}/{N}  {(time.time()-t0)/60:.1f}ph", flush=True)
    flush()
    torch.save(out, VIS)
    print(f"ảnh: {n_ok:,} trích, {n_bad} hỏng, {N-n_ok:,} vector 0 -> {tuple(out.shape)}")

In [ ]:
# ĐẶC TRƯNG VĂN BẢN — multilingual-e5-large (1024-d), mean-pooling có mask.
# e5 yêu cầu tiền tố "passage: ".
TXT = f"{OUT}/ViMMKG-textual.pth"
if os.path.exists(TXT):
    outt = torch.load(TXT, map_location="cpu")
    print(f"đã có: {tuple(outt.shape)}")
else:
    from transformers import AutoTokenizer, AutoModel
    tk = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
    mt = AutoModel.from_pretrained("intfloat/multilingual-e5-large").eval().to(DEV)
    ex = json.load(open(f"{EX}/viwiki_extracts.json"))
    ti = json.load(open(f"{TI}/viwiki_titles.json"))
    outt = torch.zeros(N, mt.config.hidden_size)
    idx, txt = [], []; n_ok=0; t0=time.time(); BS=32
    @torch.no_grad()
    def flushT():
        global idx, txt
        if not txt: return
        b = tk(txt, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEV)
        h = mt(**b).last_hidden_state
        msk = b["attention_mask"].unsqueeze(-1).float()
        outt[idx] = ((h*msk).sum(1)/msk.sum(1)).float().cpu()
        idx, txt = [], []
    for i,q in enumerate(ents):
        t = ex.get(q)
        if not t:
            lb = (ti.get(q) or {}).get("label")
            if not lb: continue
            t = lb
        txt.append("passage: "+t[:2000]); idx.append(i); n_ok+=1
        if len(txt)>=BS:
            flushT()
            if n_ok % 2048 < BS: print(f"  {n_ok:>6}/{N}  {(time.time()-t0)/60:.1f}ph", flush=True)
    flushT()
    torch.save(outt, TXT)
    print(f"văn bản: {n_ok:,} trích, {N-n_ok:,} vector 0 -> {tuple(outt.shape)}")

In [ ]:
# missing.json ĐÚNG theo dữ liệu thật.
# Lưu ý: missing.json trong kho mã APKGC liệt kê 2 và 1 chỉ số cho MKG-W trong
# khi embedding MKG-W thật có 537 và 851 hàng toàn 0 — nên cách xử lý mô thái
# thiếu ở đây KHÔNG so sánh trực tiếp được với số MKG-W đã công bố.
miss = {}
for mod, t in (("visual", out), ("textual", outt)):
    z = (t.abs().sum(1)==0).nonzero().flatten().tolist()
    miss[f"ViMMKG-{mod}.pth"] = z
    print(f"ViMMKG-{mod}.pth: {len(z):,}/{N:,} hàng toàn 0 ({len(z)/N*100:.1f}%)")
json.dump(miss, open(f"{OUT}/missing_ViMMKG.json","w"))

# nén ảnh thành MỘT tệp để cập nhật dataset sau này
import tarfile
TAR = f"{OUT}/images_full.tar.gz"
if not os.path.exists(TAR):
    with tarfile.open(TAR, "w:gz") as tar: tar.add(f"{W}/images", arcname="images")
    print(f"đã nén ảnh -> {TAR}")
print("\nTRÊN DRIVE:", sorted(os.listdir(OUT)))